# 05 · DKT: the first neural tracer
**Kuizi brain walkthrough, notebook 5 of 10**

**Deep Knowledge Tracing** (DKT, 2015) replaces BKT's four numbers with a **neural network**
that reads the answers one by one and keeps a summary of the student in a memory vector (the
*hidden state* of an **LSTM**, a network designed to remember over long sequences).

- Input at each step: which concept, and right or wrong.
- Output at each step: the chance of success on **every** concept at the next step.

Because one memory serves all concepts, DKT can learn that success on *Dérivées* predicts
success on *Dipôle RC*. It is the strongest classic baseline in the T040 benchmark.

<div dir="rtl" lang="ar">

## ٠٥ · DKT: أوّل متتبّع بشبكة عصبية

<b>تتبّع المعرفة العميق</b> (DKT) يستبدل أرقام BKT الأربعة بـ<b>شبكة عصبية</b> تقرأ الإجابات واحدة بعد الأخرى وتحتفظ
بملخّص عن التلميذ في متّجه ذاكرة (الحالة المخفية لشبكة <b>LSTM</b>، المصمّمة لتتذكّر عبر المتتاليات الطويلة).

المدخل في كل خطوة: أي مفهوم، وصحيح أم خاطئ. المخرج: احتمال النجاح في <b>كل</b> المفاهيم في الخطوة التالية.
لأن ذاكرة واحدة تخدم كل المفاهيم، يستطيع DKT أن يتعلّم أن النجاح في الاشتقاق يتنبّأ بالنجاح في RC.

</div>

In [ ]:
# Concept: training DKT and BKT on real answers · Production: T040, T041
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from kuizi_brain import data as D, models as M, metrics as X
df = D.load_assist2009()
train, test = D.split_by_student(df)
w_train, w_test = M.make_windows(train), M.make_windows(test)
n_kc = df.kc.max() + 1
dkt, bkt = M.DKT(n_kc), M.BKT(n_kc)
print("DKT"); M.fit(dkt, w_train, epochs=6)
print("BKT"); M.fit(bkt, w_train, epochs=4, lr=0.05)
rows = [X.report(name, *M.predict(model, w_test)) for name, model in [("BKT", bkt), ("DKT", dkt)]]
print(f"DKT has {M.count_parameters(dkt):,} parameters; BKT has {M.count_parameters(bkt):,}")
pd.DataFrame(rows)

## Looking inside: DKT's view of one simulated student

On the simulator we know the truth. We train DKT on simulated Bac Physics students, then show
what it believes about **all 12 concepts** after each answer of one test student.

<div dir="rtl" lang="ar">

## نظرة إلى الداخل: كيف يرى DKT تلميذًا افتراضيًا

في المحاكي نعرف الحقيقة. ندرّب DKT على تلاميذ فيزياء افتراضيين، ثم نعرض ما يعتقده عن <b>المفاهيم الاثني عشر كلها</b>
بعد كل إجابة لتلميذ من مجموعة الاختبار.

</div>

In [ ]:
# Concept: DKT's estimate on every concept after each answer · Production: T040, T049
sim = D.simulate_students(n_students=1000, answers_per_student=120, seed=2)
s_train, s_test = D.split_by_student(sim)
dkt_sim = M.DKT(sim.attrs["n_kc"])
M.fit(dkt_sim, M.make_windows(s_train), epochs=8, verbose=False)
student = s_test.student.iloc[0]
hist = s_test[s_test.student == student].head(100)
view = dkt_sim.predict_all(M.make_windows(hist))[0, :len(hist)].numpy()
kc_names = sim.groupby("kc").kc_name.first()
fig, ax = plt.subplots(figsize=(12, 4.5))
im = ax.imshow(view.T, aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax.set_yticks(range(len(kc_names))); ax.set_yticklabels(kc_names, fontsize=8)
ax.scatter(np.arange(len(hist)), hist.kc.values, c=["black" if c else "white" for c in hist.correct], edgecolors="black", s=14, marker="o")
ax.set_xlabel("answer number (dot = concept practised; filled = right, hollow = wrong)")
fig.colorbar(im, label="DKT's chance of success")
ax.set_title("DKT updates its view of every concept, not only the one practised")
plt.show()

In [ ]:
# Concept: does DKT follow the true chance of success? · Production: T040, T047
w_test_sim = M.make_windows(s_test)
y, p = M.predict(dkt_sim, w_test_sim)
corr = np.corrcoef(p, s_test.p_true)[0, 1]
print(f"AUC on simulated test students: {X.auc(y, p):.3f}")
print(f"Best possible AUC (knowing the hidden truth): {X.auc(s_test.correct, s_test.p_true):.3f}")
print(f"Correlation between DKT's prediction and the true chance of success: {corr:.2f}")

Even a perfect model cannot reach AUC 1.0: a student who has a 70 % chance still fails 3 times
out of 10. The "best possible AUC" is the ceiling set by luck.

### Limits of DKT
- Its memory is a black box: it cannot say *which* past answers mattered.
- It reads strictly in order, so a key answer from long ago may fade from memory.
- It ignores item difficulty and time.

SAINT+ addresses all three with **attention** (next notebook).

### Remember
- DKT = an LSTM that reads (concept, right/wrong) pairs and predicts success on every concept.
- One shared memory lets it link concepts, which BKT cannot do.
- Strong baseline, but not explainable; Kuizi needs to explain its estimates (T049).

<div dir="rtl" lang="ar">

حتى النموذج المثالي لا يبلغ AUC يساوي 1.0: التلميذ الذي له فرصة 70% يفشل 3 مرّات من 10. "أفضل AUC ممكن" هو السقف الذي يحدّده الحظ.

### حدود DKT
- ذاكرته صندوق أسود: لا يستطيع أن يقول <b>أي</b> الإجابات السابقة كانت مهمّة.
- يقرأ بالترتيب فقط، فقد تتلاشى إجابة مهمّة قديمة من ذاكرته.
- يتجاهل صعوبة التمارين والزمن.

يعالج SAINT+ هذه الحدود الثلاثة بـ<b>آلية الانتباه</b> (الدفتر التالي).

</div>